In [1]:
import pandas as pd

df = pd.read_csv("data/processed/flood_rainfall_features_2022.csv")

print(df.shape)
print(df.columns.tolist())

(267364, 10)
['State', 'District', 'month', 'Day', 'Rainfall', 'Date', 'flood_occurred', 'rainfall_3day', 'rainfall_7day', 'historical_flood_count_10yr']


In [2]:
X = df.drop(columns=["flood_occurred", "Date"])
y = df["flood_occurred"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (267364, 8)
y shape: (267364,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (213891, 8)
X_test: (53473, 8)
y_train: (213891,)
y_test: (53473,)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

categorical_features = ["State", "District"]

numeric_features = [
    "month",
    "Day",
    "Rainfall",
    "rainfall_3day",
    "rainfall_7day",
    "historical_flood_count_10yr"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight="balanced",
    max_iter=300,
    solver="liblinear",
    random_state=42
)

print("Model created.")

Model created.


In [8]:
from sklearn.pipeline import Pipeline

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Pipeline created.")

Pipeline created.


In [9]:
lr_pipeline.fit(X_train, y_train)

print("Training completed.")

Training completed.


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = lr_pipeline.predict(X_test)
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.87      0.93     53277
           1       0.02      0.72      0.04       196

    accuracy                           0.87     53473
   macro avg       0.51      0.80      0.49     53473
weighted avg       1.00      0.87      0.93     53473


Confusion Matrix:
[[46553  6724]
 [   54   142]]

ROC-AUC: 0.8889900799556266


In [11]:
from sklearn.ensemble import RandomForestClassifier

print("Random Forest imported successfully.")

Random Forest imported successfully.


In [12]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully.")

Random Forest model created successfully.


In [13]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_model)
])

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [14]:
rf_pipeline.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [15]:
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print("Predictions completed.")

Predictions completed.


In [16]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Classification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     53277
           1       0.89      0.08      0.15       196

    accuracy                           1.00     53473
   macro avg       0.94      0.54      0.57     53473
weighted avg       1.00      1.00      1.00     53473


Confusion Matrix:
[[53275     2]
 [  180    16]]

ROC-AUC: 0.8669577043047636


In [17]:
y_prob = lr_pipeline.predict_proba(X_test)[:, 1]

print(y_prob[:10])


[5.34195245e-04 1.84993418e-03 2.32308368e-03 2.72968699e-01
 1.08518713e-03 4.29859144e-02 2.59305166e-05 5.79197522e-01
 5.99209818e-03 1.17441784e-01]


In [18]:
import numpy as np

threshold = 0.50

y_pred_50 = (y_prob >= threshold).astype(int)

print("Flood predictions:", y_pred_50.sum())

Flood predictions: 6866


In [19]:
threshold = 0.30

y_pred_30 = (y_prob >= threshold).astype(int)

print("Flood predictions:", y_pred_30.sum())

Flood predictions: 11882


In [20]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_30))

              precision    recall  f1-score   support

           0       1.00      0.78      0.88     53277
           1       0.01      0.81      0.03       196

    accuracy                           0.78     53473
   macro avg       0.51      0.80      0.45     53473
weighted avg       1.00      0.78      0.87     53473

